# 02 — Architecture (Figures 2–3)

**Tujuan.** Tunjukkan sembilan blok kandidat, shape, parameter, dan estimasi MAC. **Sumber:** §III-B–D, Pers. (1)–(6), Figure 2–3, Tabel 1 hlm. 91507–91509. `KEEP_CURRENT`: width/depth/heads/patch adalah asumsi; 5.282.101 parameter kandidat berbeda dari 7,2629 M paper.

```mermaid
flowchart LR
  S[Stem C3+BS] --> A[IRB-1] --> B[IRB+A-1] --> C[IRB-2] --> D[IRB+A-2] --> E[EMB-1] --> F[IRB+A-3] --> G[EMB-2] --> H[IRB+A-4] --> I[EMB-3] --> J[C1+BS → AP → FC]
```

EMB: MFB rates 3/5/7/9 → **Concat** (Pers. 2) → Unfold → Transformer×L → Fold → residual/projection/fusion. Diagram ini diturunkan dari kode dan paper, bukan arsitektur resmi penulis.

**Figure 2 / Persamaan (2):** Feature map → empat cabang dilated 3×3 (rate 3, 5, 7, 9) → Concat channels → Conv 3×3 → output. Diagram paper dapat tampak memakai node penjumlahan; Persamaan (2) menuliskan Concat, yang dipakai kode.


### Struktur komponen aktual (`reproduction/cadvit/model.py`)

| Komponen | Jalur dan konfigurasi kandidat | Kaitan paper / status |
| --- | --- | --- |
| Stem | `ConvBNAct(3,32,3,stride=2)` | Urutan stem pada Gambar 3; lebar kanal **asumsi** |
| IRB | Conv 1×1 ekspansi 2 → depthwise 3×3 → proyeksi 1×1; residual jika stride 1 dan kanal sama | Inverted residual Gambar 3; ekspansi/stride **asumsi** |
| IRB+A | IRB dengan `CoordinateAttention` setelah depthwise | Posisi attention kandidat berasal dari interpretasi Gambar 3 |
| Coordinate Attention | Rata-rata terpisah arah tinggi/lebar → gabung → 1×1+BN+SiLU → pecah → gerbang sigmoid H/W → kalikan fitur; `max(1,C//r)`, r=16 | Pers. (3)–(6); pembulatan, aktivasi, dan lokasi tepat **asumsi** |
| MFB | Empat Conv 3×3 dilation 3/5/7/9 → concat kanal → Conv 3×3 fusion | Figure 2 dan Pers. (1)–(2); pilihan detail kanal implementasi |
| EMB | MFB → patch 2×2 `F.unfold` → dua `TransformerEncoderLayer`, 4 heads → `F.fold` → proyeksi 1×1 dan fusion 3×3 dengan residual | Gambar 3; patch/depth/heads **asumsi** |
| Head | Conv 1×1 ke 320 kanal → adaptive average pool → linear 5 kelas | Head Gambar 3; kanal/head **asumsi** |

**Diagram komponen kandidat (source-level):**

```mermaid
flowchart LR
  subgraph IRB[IRB dan IRB+A]
    A[Input] --> B[1×1 expansion 2×] --> C[Depthwise 3×3]
    C --> D[Coordinate Attention jika IRB+A] --> E[1×1 projection] --> F[Residual jika shape sama]
  end
  subgraph CA[Coordinate Attention]
    G[Feature map] --> H[Mean sepanjang width / height] --> I[Concat + 1×1 BN SiLU] --> J[Split H dan W] --> K[Sigmoid gates] --> L[Kalikan input]
  end
  subgraph MFB[MFB]
    M[Feature map] --> N[Empat Conv 3×3: dilation 3,5,7,9] --> O[Concat kanal] --> P[Conv 3×3 fusion]
  end
  subgraph EMB[Enhanced MobileViT Block]
    Q[Input] --> R[MFB] --> S[Unfold patch 2×2] --> T[Transformer 2 layer, 4 heads] --> U[Fold] --> V[1×1 project + residual] --> W[3×3 fusion]
  end
```

Diagram di atas menyajikan urutan sembilan blok; shape disimpan di output **SYNTHETIC_TEST** 64×64 berikutnya. Input eksperimen adalah 512×512, sehingga dimensi spasial output sintetis tidak boleh diperlakukan sebagai ukuran feature map eksperimen. Nilai shape memeriksa konsistensi kode, bukan bobot terlatih atau performa klinis.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
import torch
from reproduction.cadvit.model import CADViTCandidate, ARCH_CONFIG
from reproduction.cadvit.experiments import core_macs
torch.set_num_threads(1)
model = CADViTCandidate().eval()
with torch.inference_mode():
    _, shapes = model.forward_features(torch.zeros(1, 3, 64, 64), record_shapes=True)
    logits = model(torch.zeros(1, 3, 64, 64))
print("SYNTHETIC_TEST input 1×3×64×64; logits", tuple(logits.shape))
for name, shape in shapes.items():
    print(name, shape)
print("Parameters total:", sum(p.numel() for p in model.parameters()))
print("Parameters by block:", [(name, sum(p.numel() for p in block.parameters())) for name, block in zip(model.block_names, model.blocks)])
print("Analytic 512×512 core MAC (not paper profiler):", core_macs(ARCH_CONFIG)["core_macs"])

SYNTHETIC_TEST input 1×3×64×64; logits (1, 5)
stem (1, 32, 32, 32)
IRB-1 (1, 32, 32, 32)
IRB+A-1 (1, 64, 16, 16)
IRB-2 (1, 64, 16, 16)
IRB+A-2 (1, 96, 8, 8)
EMB-1 (1, 96, 8, 8)
IRB+A-3 (1, 128, 4, 4)
EMB-2 (1, 128, 4, 4)
IRB+A-4 (1, 160, 2, 2)
EMB-3 (1, 160, 2, 2)
head (1, 320, 2, 2)
Parameters total: 5282101
Parameters by block: [('IRB-1', 4992), ('IRB+A-1', 8008), ('IRB-2', 18176), ('IRB+A-2', 25680), ('EMB-1', 905664), ('IRB+A-3', 53080), ('EMB-2', 1608960), ('IRB+A-4', 90208), ('EMB-3', 2512960)]
Analytic 512×512 core MAC (not paper profiler): 8952383040


### Parameter dan kesetiaan arsitektur

| Blok kandidat | Parameter dari output struktural |
| --- | ---: |
| IRB-1 / IRB+A-1 / IRB-2 / IRB+A-2 | 4.992 / 8.008 / 18.176 / 25.680 |
| EMB-1 / EMB-2 / EMB-3 | 905.664 / 1.608.960 / 2.512.960 |
| IRB+A-3 / IRB+A-4 | 53.080 / 90.208 |
| **Seluruh CADViTCandidate (r=16)** | **5.282.101** |
| **Paper Tabel 1 (r=16)** | **7.262.900** |

Selisih nominal **1.980.799 parameter** menunjukkan kandidat ini bukan rekonstruksi arsitektur identik. MAC analitik kandidat 512×512 pada output (8,952383 G) memakai konvensi yang berbeda dari FLOPs 1,9703 G paper; angka itu tidak dibandingkan sebagai rasio efisiensi. Perbedaan parameter harus diselidiki melalui spesifikasi kanal, kedalaman, operasi, dan source penulis; menambah layer hanya agar total mendekati paper akan mengubah hipotesis arsitektur tanpa bukti. [Audit kesetiaan](../../docs/ARCHITECTURE_FIDELITY_AUDIT.md) memisahkan informasi eksplisit dari asumsi.

**Interpretasi dan keterbatasan.** Forward di atas hanya sintetis 64×64; estimasi MAC memakai input 512×512 dan mengecualikan sejumlah operasi. Angka FLOPs paper 1,9703 G memakai konvensi yang tidak dijelaskan; jangan menyamakan langsung. Jumlah parameter tidak diubah agar mendekati paper.